In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Streaming-Titles-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
print("A ler a tabela 'projeto.amazon_credits'...")

df_amazon = spark.read.table("projeto.amazon_credits")

# Adiciona a coluna 'platform' com o valor 'amazon'
df_amazon_platform = df_amazon.withColumn("platform", lit("amazon"))

print(f"Registos lidos da Amazon: {df_amazon_platform.count()}")
df_amazon_platform.select("person_id", "name", "character","platform").show(5, truncate=False)

A ler a tabela 'projeto.amazon_credits'...
Registos lidos da Amazon: 124235
+---------+--------------+---------+--------+
|person_id|name          |character|platform|
+---------+--------------+---------+--------+
|28732    |clyde bruckman|na       |amazon  |
|21174    |buster keaton |na       |amazon  |
|13717    |william wyler |na       |amazon  |
|23839    |howard hawks  |na       |amazon  |
|35387    |nicholas ray  |na       |amazon  |
+---------+--------------+---------+--------+
only showing top 5 rows



In [3]:
print("A ler a tabela 'projeto.netflix_credits'...")

df_netflix = spark.read.table("projeto.netflix_credits")

# Adiciona a coluna 'platform' com o valor 'netflix'
df_netflix_platform = df_netflix.withColumn("platform", lit("netflix"))

print(f"Registos lidos da Netflix: {df_netflix_platform.count()}")
df_netflix_platform.select("person_id", "name", "character","platform").show(5, truncate=False)

A ler a tabela 'projeto.netflix_credits'...
Registos lidos da Netflix: 77801
+---------+---------------+-----------------------+--------+
|person_id|name           |character              |platform|
+---------+---------------+-----------------------+--------+
|3748     |robert de niro |travis bickle          |netflix |
|14658    |jodie foster   |iris steensma          |netflix |
|7064     |albert brooks  |tom                    |netflix |
|3739     |harvey keitel  |matthew 'sport' higgins|netflix |
|48933    |cybill shepherd|betsy                  |netflix |
+---------+---------------+-----------------------+--------+
only showing top 5 rows



In [4]:
print("--- Schema Amazon (com plataforma) ---")
df_amazon_platform.printSchema()

print("\n--- Schema Netflix (com plataforma) ---")
df_netflix_platform.printSchema()

--- Schema Amazon (com plataforma) ---
root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)
 |-- platform: string (nullable = false)


--- Schema Netflix (com plataforma) ---
root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)
 |-- platform: string (nullable = false)



In [5]:
df_streaming_titles = df_amazon_platform.unionByName(df_netflix_platform)

print("União concluída.")

União concluída.


In [6]:
count_amazon = df_amazon_platform.count()
count_netflix = df_netflix_platform.count()
count_total = df_streaming_titles.count()

print(f"Registos Amazon:   {count_amazon}")
print(f"Registos Netflix:  {count_netflix}")
print(f"Registos Totais:   {count_total}")

is_match = (count_amazon + count_netflix) == count_total
print(f"\nVerificação (Soma == Total?): {is_match}")

if not is_match:
    print("ALERTA: A soma das contagens não bate com o total!")

Registos Amazon:   124235
Registos Netflix:  77801
Registos Totais:   202036

Verificação (Soma == Total?): True


In [7]:
print("Contagem final por plataforma:")
df_streaming_titles.groupBy("platform").count().show()

print("\nSchema final unificado:")
df_streaming_titles.printSchema()

print("\nAmostra de dados unificados:")
df_streaming_titles.show(10)

Contagem final por plataforma:
+--------+------+
|platform| count|
+--------+------+
|  amazon|124235|
| netflix| 77801|
+--------+------+


Schema final unificado:
root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)
 |-- platform: string (nullable = false)


Amostra de dados unificados:
+---------+--------+---------------+---------+--------+--------+
|person_id|      id|           name|character|    role|platform|
+---------+--------+---------------+---------+--------+--------+
|    28732| tm19248| clyde bruckman|       na|director|  amazon|
|    21174| tm19248|  buster keaton|       na|director|  amazon|
|    13717| tm82253|  william wyler|       na|director|  amazon|
|    23839| tm83884|   howard hawks|       na|director|  amazon|
|    35387| tm56584|   nicholas ray|       na|director|  amazon|
|    25637|tm160494|      john ford|       na|dir

In [8]:
df_streaming_titles \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/streaming_credits")

In [9]:
df_streaming_titles.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("role") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/streaming_credits") \
    .saveAsTable("projeto.streaming_credits")

In [10]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.streaming_credits
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2025-11-17 21:32:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -> 4, n...|        null|Apache-Spark/3.4....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

In [11]:
spark.sql(
    """
    SELECT * FROM projeto.streaming_credits
    """
).show()

+---------+-------+------------------+--------------------+-----+--------+
|person_id|     id|              name|           character| role|platform|
+---------+-------+------------------+--------------------+-----+--------+
|     3748|tm84618|    robert de niro|       travis bickle|actor| netflix|
|    14658|tm84618|      jodie foster|       iris steensma|actor| netflix|
|     7064|tm84618|     albert brooks|                 tom|actor| netflix|
|     3739|tm84618|     harvey keitel|matthew 'sport' h...|actor| netflix|
|    48933|tm84618|   cybill shepherd|               betsy|actor| netflix|
|    32267|tm84618|       peter boyle|              wizard|actor| netflix|
|   519612|tm84618|    leonard harris|senator charles p...|actor| netflix|
|    29068|tm84618|    diahnne abbott|     concession girl|actor| netflix|
|   519613|tm84618|       gino ardito|  policeman at rally|actor| netflix|
|     3308|tm84618|   martin scorsese|passenger watchin...|actor| netflix|
|    43791|tm84618|     m

In [12]:
spark.stop()